# 번호판 글자 인식 모델 준비 (Colab → Jetson Nano)

PaddleOCR **한국어 글자 인식 모델(PP-OCRv3)** 을 ONNX로 변환합니다.
번호판 위치는 규칙 기반(`plate_detect.py`)으로 찾으므로 **인식 모델 하나만** 필요합니다.

| 만들어지는 파일 | 용도 |
|---|---|
| `models/rec.onnx` | 한국어 글자 인식 모델 |
| `models/korean_dict.txt` | 모델 출력 번호 → 글자 변환표 |

**격리 환경을 쓰는 이유**
- Colab 기본 Python(3.12 이상)에는 구버전 변환 도구가 설치되지 않음
- Jetson Nano에 설치할 onnxruntime(**1.11**)과 같은 버전으로 여기서 미리 실행해봐서,
  Jetson에 옮긴 뒤 "모델 버전이 안 맞아 못 읽는" 문제를 사전에 차단

런타임은 **CPU로 충분**합니다. 위에서부터 순서대로 실행하세요. (전체 약 5분)

## 1. 변환용 격리 환경 만들기

Python 3.9 전용 환경을 따로 만들어 변환 도구와 onnxruntime 1.11을 설치합니다.

In [ ]:
!pip install -q uv
!uv venv --python 3.9 /content/p2o
!uv pip install --python /content/p2o/bin/python \
    "paddlepaddle>=2.5,<3" "paddle2onnx<2" "onnx<1.17" \
    "onnxruntime==1.11.1" "numpy<2" "opencv-python-headless<4.11" setuptools

# 설치 확인: 버전 네 줄이 출력되면 성공
!/content/p2o/bin/python -c "import paddle, paddle2onnx, onnx, onnxruntime as ort; \
print('paddle', paddle.__version__); print('paddle2onnx', paddle2onnx.__version__); \
print('onnx', onnx.__version__); print('onnxruntime', ort.__version__)"

## 2. 인식 모델과 사전 다운로드

In [ ]:
%cd /content
!mkdir -p paddle_models models

# 한국어 글자 인식 모델 (PP-OCRv3)
!wget -q -O paddle_models/korean_rec.tar \
    https://paddleocr.bj.bcebos.com/PP-OCRv3/multilingual/korean_PP-OCRv3_rec_infer.tar
!tar -xf paddle_models/korean_rec.tar -C paddle_models
!ls -l paddle_models/korean_PP-OCRv3_rec_infer

# 한국어 사전 (저장소 브랜치가 바뀌어도 받을 수 있도록 여러 곳을 차례로 시도)
!for b in release/2.7 release/2.6 main; do \
    wget -q -O models/korean_dict.txt \
    https://raw.githubusercontent.com/PaddlePaddle/PaddleOCR/$b/ppocr/utils/dict/korean_dict.txt \
    && echo "사전 받음: $b" && break; done
!wc -l models/korean_dict.txt

## 3. ONNX 변환

`opset_version 11`: Jetson의 onnxruntime 1.11이 확실히 지원하는 버전으로 고정

In [ ]:
!/content/p2o/bin/paddle2onnx \
    --model_dir paddle_models/korean_PP-OCRv3_rec_infer \
    --model_filename inference.pdmodel \
    --params_filename inference.pdiparams \
    --save_file models/rec.onnx \
    --opset_version 11 \
    --enable_onnx_checker True

!ls -lh models

## 4. Jetson과 같은 onnxruntime 1.11로 검증

아래 세 항목이 모두 `OK`면 Jetson에서도 그대로 동작합니다.

In [ ]:
%%writefile /content/check.py
import numpy as np
import onnx
import onnxruntime as ort

model = onnx.load("models/rec.onnx")
ir = model.ir_version
opset = max(o.version for o in model.opset_import if o.domain in ("", "ai.onnx"))
print("[1] 모델 형식  IR {}, opset {}  ->  {}".format(
    ir, opset, "OK" if ir <= 8 and opset <= 15 else "주의: onnxruntime 1.11이 못 읽을 수 있음"))

sess = ort.InferenceSession("models/rec.onnx", providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0]
out = sess.run(None, {inp.name: np.zeros((1, 3, 48, 320), np.float32)})[0]
print("[2] 실행       입력 {} {} -> 출력 {}  ->  OK".format(inp.name, inp.shape, out.shape))

with open("models/korean_dict.txt", encoding="utf-8") as f:
    n_dict = sum(1 for _ in f)
n_out = out.shape[-1]
if n_out == n_dict + 2:
    verdict = "OK (blank + 사전 + 공백)"
elif n_out == n_dict + 1:
    verdict = "OK (blank + 사전, plate_rec.py가 자동 처리)"
else:
    verdict = "불일치: 사전 파일이 모델과 다른 버전"
print("[3] 사전       모델 출력 {}개 / 사전 {}자  ->  {}".format(n_out, n_dict, verdict))

In [ ]:
!cd /content && /content/p2o/bin/python check.py

## 5. (선택) 번호판 이미지로 실제 인식 테스트

Jetson에서 `python3 plate_detect.py 사진.jpg`를 실행하면 펴진 번호판이 `plate_0.jpg`로 저장됩니다.
그 이미지와 `plate_rec.py`를 함께 업로드하면, Jetson과 같은 코드·같은 onnxruntime 버전으로 인식해봅니다.

번호판 이미지가 아직 없으면 이 단계는 건너뛰세요.

In [ ]:
%cd /content
from google.colab import files
uploaded = files.upload()      # plate_rec.py + 번호판 이미지(plate_0.jpg 등) 선택

In [ ]:
!cd /content && for f in *.jpg *.jpeg *.png; do [ -f "$f" ] && \
    echo "== $f" && /content/p2o/bin/python plate_rec.py "$f" --crop; done

## 6. 다운로드

`models.zip`을 Jetson의 프로젝트 폴더(예: `~/smartparking/IP/picam/`)에 풀면 `models/` 폴더가 생깁니다.

In [ ]:
%cd /content
!rm -f models.zip && zip -r models.zip models/rec.onnx models/korean_dict.txt
from google.colab import files
files.download("/content/models.zip")